In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
import re
import sys

In [2]:
sys.path.append("..")
sys.path.append(".")

In [54]:
from src.data.clean_text import TextCleaner
from src.data.preproces_dataset import TextCleanTransformer

ModuleNotFoundError: No module named 'spacy'

In [4]:
print(sys.path)

['/Users/janas/Documents/Python_Projects/DisasterTweets/notebooks', '/Library/Frameworks/Python.framework/Versions/3.8/lib/python38.zip', '/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8', '/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/lib-dynload', '', '/Users/janas/.local/share/virtualenvs/DisasterTweets-4PiDNLHM/lib/python3.8/site-packages', '/Users/janas/.local/share/virtualenvs/DisasterTweets-4PiDNLHM/lib/python3.8/site-packages/IPython/extensions', '/Users/janas/.ipython', '..', '.']


In [42]:
path = "/Users/janas/Documents/Python_Projects/DisasterTweets/data/"
df = pd.read_csv(path+"train.csv")

In [43]:
df['target'].value_counts()

0    4342
1    3271
Name: target, dtype: int64

## Train test split

In [44]:
df_train, df_test = train_test_split(df, train_size=0.8, stratify=df['target'])

In [45]:
cleaner_train = TextCleaner(df_train, 'text')

In [46]:
X_train = cleaner_train.tf_tokenize(dataset='train')

In [47]:
cleaner_test = TextCleaner(df_test, 'text')

In [48]:
X_test = cleaner_test.tf_tokenize(dataset='test')

## Training model 

In [49]:
vocab = 8000
# oov = '<OOV>'
embedding = 32
# padding = 'post'
# truncate = 'post'
maxlength = 157

In [50]:
tf.keras.backend.clear_session()
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab, embedding, input_length=maxlength),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(10, activation='relu'),
#      tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding (Embedding)        (None, 157, 32)           256000    
_________________________________________________________________
global_average_pooling1d (Gl (None, 32)                0         
_________________________________________________________________
dense (Dense)                (None, 10)                330       
_________________________________________________________________
dense_1 (Dense)              (None, 1)                 11        
Total params: 256,341
Trainable params: 256,341
Non-trainable params: 0
_________________________________________________________________


In [51]:
val_labels = np.array(df_test['target'])
train_label = np.array(df_train['target'])

In [52]:
model_check = tf.keras.callbacks.ModelCheckpoint('model.h5',save_best_only = True)

In [53]:
num_epochs = 20
history = model.fit(X_train, train_label, epochs=num_epochs, validation_data=(X_test, val_labels),callbacks = [model_check])

Epoch 1/20
191/191 [==============================] - 2s 7ms/step - loss: 0.6830 - accuracy: 0.5703 - val_loss: 0.6779 - val_accuracy: 0.5706
Epoch 2/20
191/191 [==============================] - 1s 6ms/step - loss: 0.6711 - accuracy: 0.5704 - val_loss: 0.6575 - val_accuracy: 0.5739
Epoch 3/20
191/191 [==============================] - 1s 7ms/step - loss: 0.6228 - accuracy: 0.6724 - val_loss: 0.5841 - val_accuracy: 0.7420
Epoch 4/20
191/191 [==============================] - 1s 7ms/step - loss: 0.5375 - accuracy: 0.7696 - val_loss: 0.5246 - val_accuracy: 0.7617
Epoch 5/20
191/191 [==============================] - 1s 6ms/step - loss: 0.4662 - accuracy: 0.8051 - val_loss: 0.4895 - val_accuracy: 0.7814
Epoch 6/20
191/191 [==============================] - 1s 6ms/step - loss: 0.4119 - accuracy: 0.8368 - val_loss: 0.4533 - val_accuracy: 0.8037
Epoch 7/20
191/191 [==============================] - 1s 6ms/step - loss: 0.3683 - accuracy: 0.8573 - val_loss: 0.4388 - val_accuracy: 0.8017
Epoch 

In [34]:
# Plotting Learing Curves
import matplotlib.pyplot as plt


def plot_graphs(history, string):
    plt.plot(history.history[string])
    plt.plot(history.history['val_'+string])
    plt.xlabel("Epochs")
    plt.ylabel(string)
    plt.legend([string, 'val_'+string])
    plt.show()
  
plot_graphs(history, "accuracy")
plot_graphs(history, "loss")

ModuleNotFoundError: No module named 'matplotlib'

## Predictions

In [35]:
predictions = model.predict(X_test)

In [36]:
pred = []
for i in range(0,len(predictions)):
    if predictions[i][0]>0.5:
         pred.append(1)
    else:
        pred.append(0)

In [38]:
df_test['pred'] = pd.Series(pred)

<ipython-input-38-8c532ef5028a>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test['pred'] = pd.Series(pred)


In [39]:
df_test.head()

,id,keyword,location,text,target,pred
178,254,ambulance,Happily Married with 2 kids,AMBULANCE SPRINTER AUTOMATIC FRONTLINE VEHICLE...,0,0.0
4536,6450,injured,Tropical SE FLorida,#WakeUpFlorida... #Floridians more likely to b...,0,NaN
6824,9774,trapped,Utah,Hollywood movie about trapped miners released ...,0,NaN
2599,3729,destroyed,"North Port, FL",@ryanoss123 No worries you'd have to be on eve...,0,NaN
680,982,blazing,"Pig Symbol, Alabama",Montgomery come for the blazing hot weather......,1,1.0


In [40]:
df_test['pred'].isnull().sum()

1217

In [41]:
df_test.shape

(1523, 6)

In [5]:
df[~df['location'].isnull()].head()

,id,keyword,location,text,target
31,48,ablaze,Birmingham,@bbcmtd Wholesale Markets ablaze http://t.co/l...,1
32,49,ablaze,Est. September 2012 - Bristol,We always try to bring the heavy. #metal #RT h...,0
33,50,ablaze,AFRICA,#AFRICANBAZE: Breaking news:Nigeria flag set a...,1
34,52,ablaze,"Philadelphia, PA",Crying out for more! Set me ablaze,0
35,53,ablaze,"London, UK",On plus side LOOK AT THE SKY LAST NIGHT IT WAS...,0


In [10]:
df['keyword'].nunique()

221

In [15]:
df.groupby(['target', 'keyword'])['text'].count()

target  keyword            
0       ablaze                 23
        accident               11
        aftershock             34
        airplane%20accident     5
        ambulance              18
                               ..
1       wounded                26
        wounds                 10
        wreck                   7
        wreckage               39
        wrecked                 3
Name: text, Length: 438, dtype: int64

## Modeling

In [17]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['target'], test_size=0.2, random_state=42, stratify=df['target'])

In [21]:
vocab = 1000
oov = '<OOV>'
embedding = 32
padding = 'post'
truncate = 'post'
maxlength = max([len(x) for x in X_train])

In [22]:
tokenizer = keras.preprocessing.text.Tokenizer(num_words = vocab, oov_token=oov)

In [23]:
tokenizer.fit_on_texts(X_train)

In [24]:
word_index = tokenizer.word_index

In [27]:
training = tokenizer.texts_to_sequences(X_train)

In [41]:
a = ['Our Deeds are the Reason', 'of this #earthquake May ALLAH Forgive us all']

In [42]:
tokenizer.texts_to_sequences(a)

[[115, 1, 26, 5, 1], [9, 22, 265, 145, 1, 1, 88, 43]]

In [31]:
training_pad = keras.preprocessing.sequence.pad_sequences(training, maxlen=maxlength, padding=padding, truncating=truncate)

In [35]:
len(training_pad[1])

157